low api pytorch operation 

In [1]:
import random as rand
from typing import cast

import torch
from sklearn.model_selection import train_test_split

INPUT_NUMBER = 10
DATASET_SIZE = 100
torch.manual_seed(1)
rand.seed(1)
# create dataset
#
X = []
Y = []
for _ in range(DATASET_SIZE):
    x = float(rand.randint(1, 100))
    X.append(x)
    Y.append(x * 5 + rand.randint(0, 3))

x_all = torch.tensor(X, dtype=torch.float32)
y_all = torch.tensor(Y, dtype=torch.float32)

x_train_raw, x_temp_raw, y_train, y_temp =cast(
    tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor],
    train_test_split(x_all, y_all, test_size=0.30, random_state=1)
)
x_val_raw, x_test_raw, y_val, y_test =cast(
    tuple[torch.Tensor, torch.Tensor, torch.Tensor, torch.Tensor],
    train_test_split(x_temp_raw, y_temp, test_size=0.50, random_state=1)
)


x_mean, x_std = x_train_raw.mean(), x_train_raw.std()
x_train = (x_train_raw - x_mean) / x_std
x_val   = (x_val_raw   - x_mean) / x_std
x_test  = (x_test_raw  - x_mean) / x_std



lr = 0.1
n_epoch = 50
train_losses = []
val_losses = []

w = torch.randn(1 , 1, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

x_train = x_train.reshape(-1 , 1 )
y_train = y_train.reshape(-1 , 1 )
print(w.shape)
# w = w.reshape(1,1)
print(w.shape)


# no early stopping yer , but i will consider it in mind for future
for epoch in range(n_epoch):
    z = x_train @ w + b
    loss = ((z - y_train) ** 2).mean()
    loss.backward()
    with torch.no_grad():
        gw, gb = w.grad, b.grad
        assert gw is not None and gb is not None
        w -= lr * gw
        b -= lr * gb
        gw.zero_()
        gb.zero_()
    # Validate 
    with torch.no_grad():
        loss_val = (((x_val * w + b)-y_val)**2).mean()
        val_losses.append(loss_val)

    train_losses.append(loss)
    print(f"Epoch {epoch+1} | loss {loss.item()} | validation loss {val_losses[epoch].item()}")

test_val = (((x_test * w + b)-y_test)**2).mean()
print(f"Test Validation  {test_val}")

checkpoint = {
    "state_dict": {"w": w.detach(), "b": b.detach()},
    "scaler": {"mean": x_mean, "std": x_std},
    "config": {"lr": lr, "epoch": n_epoch, "model": "linear_1d", "n_train": len(x_train), "n_test": len(x_test)},
    "metrics": {
        "train": [t.item() for t in train_losses],
        "val":   [t.item() for t in val_losses],
        "test":  test_val.item(),
    },
    "version": "1.0.0"
}
torch.save(checkpoint, "model_v1.0.0.pt")
print(f"checkpoint \n {checkpoint}")

torch.Size([1, 1])
torch.Size([1, 1])


Epoch 1 | loss 85085.3984375 | validation loss 66656.546875
Epoch 2 | loss 54543.734375 | validation loss 42756.5859375
Epoch 3 | loss 34965.5625 | validation loss 27427.30078125
Epoch 4 | loss 22415.2265625 | validation loss 17594.93359375
Epoch 5 | loss 14369.916015625 | validation loss 11288.150390625
Epoch 6 | loss 9212.478515625 | validation loss 7242.64013671875
Epoch 7 | loss 5906.27978515625 | validation loss 4647.5185546875
Epoch 8 | loss 3786.80810546875 | validation loss 2982.7060546875
Epoch 9 | loss 2428.086181640625 | validation loss 1914.6351318359375
Epoch 10 | loss 1557.04638671875 | validation loss 1229.35693359375
Epoch 11 | loss 998.6429443359375 | validation loss 789.637451171875
Epoch 12 | loss 640.658935546875 | validation loss 507.4532775878906
Epoch 13 | loss 411.1589050292969 | validation loss 326.34033203125
Epoch 14 | loss 264.0273742675781 | validation loss 210.07798767089844
Epoch 15 | loss 169.70138549804688 | validation loss 135.42953491210938
Epoch 16 |

/home/kk_gorbee/Documents/project/ML/Ames_Housing/env/lib/python3.11/site-packages/torch/autograd/graph.py:1072: UserWarning: The detected GPU (Intel(R) UHD Graphics) is not officially supported by PyTorch XPU. Running workloads on this device may result in unexpected behavior.
For stable and fully supported execution, please use GPUs based on Intel Arc (Alchemist) series or newer.
Refer to the hardware prerequisites for more information: https://docs.pytorch.org/docs/main/notes/get_start_xpu.html#hardware-prerequisite (Triggered internally at /__w/pytorch/pytorch/c10/xpu/XPUFunctions.cpp:128.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
